# Entraînement sur GPU (Google Colab)

Entraîne une ou plusieurs configs de `configs/` sur le GPU gratuit de Colab, les évalue sur toute la
validation (20 pulses par configuration, 13 SNR), compare chaque run à son parent et copie les dossiers
`runs/` sur Google Drive.

**Avant de lancer**

1. `Exécution > Modifier le type d'exécution > GPU T4`.
2. Dépose `recorded_dataset.zip` (2,5 Go) dans ton Google Drive et renseigne son chemin ci-dessous.
3. Le dépôt est privé : prépare un jeton GitHub en lecture seule
   (`github.com/settings/tokens`, droit `Contents: Read` sur le dépôt). Il est demandé à l'exécution et
   n'est jamais écrit dans le notebook.

**Règles** : entraînement sur les pulses `train` uniquement, validation pour évaluer seulement. Ne dépose
jamais le blind test sur Drive ni dans Colab.

In [ ]:
REPO = "Nina-ghz/DATATHON.ieee"
BRANCH = "main"
DATASET_ZIP = "/content/drive/MyDrive/recorded_dataset.zip"
DRIVE_RUNS = "/content/drive/MyDrive/radarml_runs"       # les runs y sont copiés un par un

# Entraînés dans cet ordre. Chaque run est comparé à son parent s'il est aussi dans la liste.
EXPERIMENTS = ["configs/e003_lowsnr.yaml", "configs/e004_fullres.yaml", "configs/e005_fulldata.yaml"]
PARENTS = {"E004_fullres": "E003_lowsnr", "E005_fulldata": "E003_lowsnr"}

In [ ]:
import torch

!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
import os
from getpass import getpass

if not os.path.isdir("/content/repo"):
    token = getpass("Jeton GitHub (lecture seule) : ")
    !git clone --quiet --branch {BRANCH} https://{token}@github.com/{REPO}.git /content/repo
    !git -C /content/repo remote set-url origin https://github.com/{REPO}.git
    del token
%cd /content/repo
!git log --oneline -1
!pip install --quiet -r requirements.txt && pip install --quiet -e .

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_RUNS, exist_ok=True)
# Copie locale : lire 660 fichiers directement sur Drive ralentit fortement l'entraînement.
!cp -n "{DATASET_ZIP}" /content/recorded_dataset.zip
!python scripts/setup_data.py --zip /content/recorded_dataset.zip --skip-file-hashes
!python -m pytest -q

## Entraînement et évaluation

Chaque run est copié sur Drive dès qu'il est terminé. Si la session est coupée, relance les cellules du
haut puis retire de `EXPERIMENTS` les configs dont le dossier est déjà sur Drive : `train.py` repart de
zéro pour une config, il ne reprend pas un entraînement interrompu.

Temps mesurés sur CPU (4 threads) pour 8 époques : E003 63 min, E004 environ 75 min. Sur GPU, le bruit
SNR reste calculé par les 2 cœurs CPU de Colab et devient le facteur limitant ; E005 voit 4 fois plus de
pulses par époque que E003.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

import yaml

runs = {}
for config in EXPERIMENTS:
    experiment = yaml.safe_load(Path(config).read_text(encoding="utf-8"))["experiment"]
    subprocess.run([sys.executable, "scripts/train.py", "--config", config, "--device", "cuda"], check=True)
    run = sorted(Path("runs").glob(f"{experiment}_*"))[-1]
    subprocess.run([sys.executable, "scripts/evaluate.py", "--run", str(run), "--device", "cuda"], check=True)
    shutil.copytree(run, Path(DRIVE_RUNS) / run.name, dirs_exist_ok=True)
    runs[experiment] = run
    print(f"{experiment} : copié dans {DRIVE_RUNS}/{run.name}", flush=True)

## Comparaison au parent, sur toute la validation

Règle du journal : gain en score moyen d'au moins +0,010 pour retenir ; entre +0,005 et +0,010 le
résultat est provisoire et doit être répliqué.

In [ ]:
for experiment, parent in PARENTS.items():
    if experiment in runs and parent in runs:
        print("=" * 100)
        subprocess.run([sys.executable, "scripts/compare_runs.py", "--parent", str(runs[parent]),
                        "--run", str(runs[experiment]), "--full", "--snrs", "0", "-5", "-10", "-15", "-20"], check=True)

## Ensuite

Télécharge les dossiers de `MyDrive/radarml_runs/` dans le dossier `runs/` du dépôt local, puis ajoute
une ligne par run à `docs/EXPERIMENT_LOG.md`. Les temps d'inférence mesurés ici sont des temps GPU : ils
ne se comparent pas aux lignes CPU du journal.